# Business Entity Resolution: v6 (separate models per country, more training data)

**How to run:** click **Save Version** (top right), choose **Save & Run All (Commit)**, then **Save**. Settings: CPU (no accelerator). It runs on Kaggle's servers even if you close the tab.

What is new compared with v1 (the features and shortlist are the same as v1):
1. **Much more training data:** 350k India + 500k US businesses (v1: 150k in total).
2. **One model per country:** India and US each get their own model, because Indian and US records are messy in different ways.
3. **A general model for France** (France is only in the test set): trained on both countries, without the features that mean something different in France.
4. **Separate cut-offs** per country and per source (Source 2 and Source 3), each tuned for F0.5.
5. **Stricter cut-off for France** (+0.15), because v1 made many wrong France matches (public board 0.9405 vs 0.9529 on validation). Change `UNSEEN_MARGIN` in the settings cell to adjust it.
6. **Name-commonness clues:** an exact name match counts for less when many businesses share that name (common in France).

It is checked on the **same 30k held-out businesses as v1** (v1 scored 0.9529), so the numbers compare directly. Look for the lines starting with `BEST val F0.5`.

Cells: 1 install, 2 v1 pipeline code, 3 settings, 4 v6 helpers, 5 **train** (about 4 to 5 hours), 6 **predict** test and write `output/matching_results.tsv` (about 2 to 3 hours), 7 official format check. No external data or APIs are used.

In [ ]:
!pip install -q rapidfuzz sparse_dot_topn unidecode polars lightgbm

In [ ]:
# =====================================================================
# Business Entity Resolution: full pipeline (clean -> shortlist -> features -> LightGBM -> decide)
# No external data or services are used anywhere. Only the provided files.
# =====================================================================
import os, re, gc, glob, time, json
import numpy as np
import pandas as pd
import polars as pl
from multiprocessing import Pool
from unidecode import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist
import lightgbm as lgb

N_CPU = os.cpu_count() or 4

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

# ---------------------------------------------------------------------
# 1. Loading. Files are tab-separated, all columns read as text.
#    Handles files split into *_part_N.tsv (BOM / CRLF / repeated header).
# ---------------------------------------------------------------------
def find_file(root, name):
    hits = sorted(glob.glob(f"{root}/**/{name}.tsv", recursive=True))
    if hits:
        return [hits[0]]
    parts = glob.glob(f"{root}/**/{name}_part_*.tsv", recursive=True)
    parts.sort(key=lambda p: int(re.findall(r"_part_(\d+)", p)[-1]))
    if not parts:
        raise FileNotFoundError(f"{name}.tsv not found under {root}")
    return parts

def load(root, name):
    frames, header = [], None
    for p in find_file(root, name):
        txt = open(p, encoding="utf-8-sig").read().replace("\r\n", "\n").replace("\r", "")
        if txt.startswith(("entity_id\t", "source1_entity_id\t")):
            header, txt = txt.split("\n", 1)
        frames.append(txt)
    body = "".join(f if f.endswith("\n") else f + "\n" for f in frames)
    df = pl.read_csv((header + "\n" + body).encode(), separator="\t", quote_char=None, infer_schema_length=0)
    return df.fill_null("").to_pandas()

# ---------------------------------------------------------------------
# 2. Cleaning (normalisation). Hand-written rules only.
# ---------------------------------------------------------------------
ABBR = {"rd":"road","st":"street","ave":"avenue","av":"avenue","blvd":"boulevard","bd":"boulevard","hwy":"highway",
        "dr":"drive","ln":"lane","ct":"court","pl":"place","pkwy":"parkway","cir":"circle","ste":"suite","fl":"floor",
        "apt":"apartment","no":"number","nr":"near","opp":"opposite","bldg":"building","ind":"industrial",
        "dist":"district","r":"rue","ch":"chemin","n":"north","s":"south","e":"east","w":"west","mt":"mount",
        "ft":"fort","hts":"heights","sq":"square","imp":"impasse","all":"allee"}
NULLS = {"null", "na", "none", "nan"}
# legal words, written in "skeleton" form (see skel_tok): covers Limited/Ltd/limittedd, Private/Pvt/praaivett, LLC, SARL ...
LEGAL_SK = {"lmtd","ltd","prwt","pwt","llc","llp","inc","ncrprtd","crp","crprtn","c","cmpny","plc","srl","ss","sa",
            "sc","erl","snc","gmbh","t","m","s","d","dba","lp","pc","pllc","pa","ms","and"}

def base(s):
    s = unidecode(str(s)).lower()                                   # also turns Hindi/Telugu/... script into Latin letters
    s = re.sub(r"<?null>?|n/a", " ", s)
    s = re.sub(r"https?://|www\.|\.(com|net|org|in|co|io|biz|fr|us)\b", " ", s)
    s = s.replace("&", " and ").replace("+", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def skel_tok(t):
    """Spelling-robust 'skeleton' of a word: drop vowels, merge ph/f, bh/b ..., collapse double letters.
    'limited' and the transliterated 'limittedd' both become 'lmtd'."""
    if not t or t.isdigit():
        return t
    t = t.replace("0", "o").replace("1", "l") if re.search(r"[a-z]", t) else t   # leetspeak: C1ean -> Clean
    t = re.sub(r"ph", "f", t)
    t = re.sub(r"([bkgtdcs])h", r"\1", t)
    t = t.replace("v", "w").replace("q", "k").replace("z", "s").replace("x", "ks")
    t = t[0] + re.sub(r"[aeiouy]", "", t[1:])
    return re.sub(r"(.)\1+", r"\1", t)

US = {"alabama":"al","alaska":"ak","arizona":"az","arkansas":"ar","california":"ca","colorado":"co","connecticut":"ct",
"delaware":"de","florida":"fl","georgia":"ga","hawaii":"hi","idaho":"id","illinois":"il","indiana":"in","iowa":"ia",
"kansas":"ks","kentucky":"ky","louisiana":"la","maine":"me","maryland":"md","massachusetts":"ma","michigan":"mi",
"minnesota":"mn","mississippi":"ms","missouri":"mo","montana":"mt","nebraska":"ne","nevada":"nv","new hampshire":"nh",
"new jersey":"nj","new mexico":"nm","new york":"ny","north carolina":"nc","north dakota":"nd","ohio":"oh","oklahoma":"ok",
"oregon":"or","pennsylvania":"pa","rhode island":"ri","south carolina":"sc","south dakota":"sd","tennessee":"tn",
"texas":"tx","utah":"ut","vermont":"vt","virginia":"va","washington":"wa","west virginia":"wv","wisconsin":"wi",
"wyoming":"wy","district of columbia":"dc","puerto rico":"pr"}
IN = {"andhra pradesh":"ap","arunachal pradesh":"ar","assam":"as","bihar":"br","chhattisgarh":"cg","goa":"ga",
"gujarat":"gj","haryana":"hr","himachal pradesh":"hp","jharkhand":"jh","karnataka":"ka","kerala":"kl",
"madhya pradesh":"mp","maharashtra":"mh","manipur":"mn","meghalaya":"ml","mizoram":"mz","nagaland":"nl","odisha":"od",
"orissa":"od","punjab":"pb","rajasthan":"rj","sikkim":"sk","tamil nadu":"tn","telangana":"ts","tripura":"tr",
"uttar pradesh":"up","uttarakhand":"uk","uttaranchal":"uk","west bengal":"wb","delhi":"dl","new delhi":"dl",
"jammu and kashmir":"jk","ladakh":"la","puducherry":"py","pondicherry":"py","chandigarh":"ch",
"andaman and nicobar islands":"an","lakshadweep":"ld"}
MERGE_ST = {"ts": "ap"}            # Telangana was split from Andhra Pradesh; the data mixes them
_STATE = {}
for _d in (US, IN):
    for _full, _code in _d.items():
        _STATE[tuple(skel_tok(t) for t in _full.split())] = _code
_CODES = set(US.values()) | set(IN.values())

def states_of(raw_addr):
    """All state/province codes mentioned in an address ('' when none). Country-agnostic: unknown countries get ''."""
    found = set()
    for seg in unidecode(str(raw_addr)).lower().split(","):
        toks = re.sub(r"[^a-z ]", " ", seg).split()
        if not toks:
            continue
        if len(toks) == 1 and toks[0] in _CODES:
            found.add(toks[0]); continue
        sk = [skel_tok(t) for t in toks]
        for n in (4, 3, 2, 1):
            for i in range(len(sk) - n + 1):
                key = tuple(sk[i:i + n])
                if key in _STATE and (n > 1 or len(key[0]) >= 3):
                    found.add(_STATE[key])
    return "|".join(sorted({MERGE_ST.get(x, x) for x in found}))

def _clean_rows(rows):
    out = []
    for name, addr in rows:
        nb = base(name)
        sk = [skel_tok(t) for t in nb.split()]
        core = [t for t in sk if t not in LEGAL_SK] or sk
        an = " ".join(ABBR.get(t, t) for t in base(addr).split() if t not in NULLS)
        out.append((nb, " ".join(core), "".join(core), an,
                    " ".join(skel_tok(t) for t in an.split()), states_of(addr)))
    return out

def prepare(df):
    """Adds the cleaned columns. Runs on all CPU cores."""
    rows = list(zip(df.business_name.tolist(), df.business_address.tolist()))
    step = 50_000
    with Pool(N_CPU) as pool:
        res = pool.map(_clean_rows, [rows[i:i + step] for i in range(0, len(rows), step)])
    res = [r for chunk in res for r in chunk]
    cols = ["name_b", "name_sk", "name_skc", "addr_n", "addr_sk", "states"]
    for j, c in enumerate(cols):
        df[c] = [r[j] for r in res]
    df.drop(columns=["business_address"], inplace=True)
    df["nonascii"] = df.business_name.str.contains(r"[^\x00-\x7f]", regex=True).astype(np.int8)
    df.drop(columns=["business_name"], inplace=True)
    return df.reset_index(drop=True)

# ---------------------------------------------------------------------
# 3. Shortlist ("blocking"). For each Source 1 record, look only inside the same country and
#    the same state, and keep the top-k most similar names and the top-k most similar addresses.
#    Records whose state could not be read form a separate pool per country.
# ---------------------------------------------------------------------
def _vec_name():
    return TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
def _vec_addr():
    return TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), dtype=np.float32, sublinear_tf=True)

class Index:
    """TF-IDF indexes over one pool of candidate records (one country of Source 2 or Source 3)."""
    def __init__(self, B, k_state=15, k_unk=10):
        self.B, self.k_state, self.k_unk = B, k_state, k_unk
        self.cache = {}
        st = B.states.str.split("|")
        self.members = {}
        for pos, lst in zip(range(len(B)), st):
            for s in lst:
                if s:
                    self.members.setdefault(s, []).append(pos)
        self.unknown = np.flatnonzero(B.states.to_numpy() == "")

    def _build(self, key, pos):
        if key in self.cache:
            return self.cache[key]
        pos = np.asarray(pos, dtype=np.int64)
        entry = {"pos": pos}
        for col, mk in (("name_skc", _vec_name), ("addr_sk", _vec_addr)):
            try:
                v = mk(); M = v.fit_transform(self.B[col].to_numpy()[pos])
                entry[col] = (v, M.T.tocsr())
            except ValueError:            # empty vocabulary, e.g. every address blank
                entry[col] = None
        self.cache[key] = entry
        return entry

    def query(self, A, thr=0.1, chunk=40_000):
        """Returns DataFrame(a, b, sim_name, sim_addr); a = row in A, b = row in self.B."""
        out = []
        ast = A.states.str.split("|").tolist()
        groups = {}
        for i, lst in enumerate(ast):
            for s in lst:
                groups.setdefault(s, []).append(i)
        jobs = []
        for s, rows in groups.items():
            if s == "":
                jobs.append(("__all__", np.arange(len(self.B)), rows, self.k_state))
            elif s in self.members:
                jobs.append((s, self.members[s], rows, self.k_state))
        known = [i for i, lst in enumerate(ast) if lst != [""]]
        if len(self.unknown) and known:
            jobs.append(("__unknown__", self.unknown, known, self.k_unk))
        for key, pos, rows, k in jobs:
            e = self._build(key, pos)
            rows = np.asarray(rows)
            for col, tag in (("name_skc", 0), ("addr_sk", 1)):
                if e[col] is None:
                    continue
                v, BT = e[col]
                texts = A[col].to_numpy()[rows]
                for i in range(0, len(rows), chunk):
                    C = sp_matmul_topn(v.transform(texts[i:i + chunk]), BT, top_n=k, threshold=thr,
                                       n_threads=N_CPU).tocoo()
                    out.append((rows[i + C.row], e["pos"][C.col], C.data, tag))
        if not out:
            return pd.DataFrame({"a": [], "b": [], "sim_name": [], "sim_addr": []})
        a = np.concatenate([o[0] for o in out]).astype(np.int64)
        b = np.concatenate([o[1] for o in out]).astype(np.int64)
        sim = np.concatenate([o[2] for o in out]).astype(np.float32)
        tag = np.concatenate([np.full(len(o[0]), o[3], np.int8) for o in out])
        df = pd.DataFrame({"key": a * len(self.B) + b,
                           "sim_name": np.where(tag == 0, sim, np.nan),
                           "sim_addr": np.where(tag == 1, sim, np.nan)})
        df = df.groupby("key", sort=False).max().reset_index()
        df["a"] = (df.key // len(self.B)).astype(np.int32)
        df["b"] = (df.key % len(self.B)).astype(np.int32)
        return df.drop(columns="key")

# ---------------------------------------------------------------------
# 4. Features: how similar is each (Source 1, candidate) pair?
# ---------------------------------------------------------------------
FULL = [("tset", fuzz.token_set_ratio), ("tsort", fuzz.token_sort_ratio), ("ratio", fuzz.ratio),
        ("partial", fuzz.partial_ratio), ("jw", JaroWinkler.normalized_similarity)]

def _nums(s):
    return set(re.findall(r"\d+", s))

def features(P, A, B, is_s3):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    g = lambda D, col, idx: D[col].to_numpy()[idx]
    f = pd.DataFrame({"sim_name": P.sim_name.to_numpy(), "sim_addr": P.sim_addr.to_numpy()})
    for col, pre, scorers in (("name_b", "nb", FULL), ("name_sk", "nsk", FULL),
                              ("name_skc", "nskc", [("ratio", fuzz.ratio), ("partial", fuzz.partial_ratio)]),
                              ("addr_sk", "ask", [("tset", fuzz.token_set_ratio), ("ratio", fuzz.ratio),
                                                  ("partial", fuzz.partial_ratio)])):
        x, y = g(A, col, a).tolist(), g(B, col, b).tolist()
        for nm, sc in scorers:
            f[f"{pre}_{nm}"] = cpdist(x, y, scorer=sc, workers=N_CPU).astype(np.float32)
    xa, ya = g(A, "addr_n", a), g(B, "addr_n", b)
    nl = [_nums(s) for s in xa]; nr = [_nums(s) for s in ya]
    f["num_jacc"] = np.array([len(p & q) / len(p | q) if (p | q) else -1 for p, q in zip(nl, nr)], np.float32)
    f["num_contain"] = np.array([len(p & q) / len(q) if q else -1 for p, q in zip(nl, nr)], np.float32)
    sa, sb = g(A, "states", a), g(B, "states", b)
    f["state_rel"] = np.array([-1 if (not p or not q) else (1 if set(p.split("|")) & set(q.split("|")) else 0)
                               for p, q in zip(sa, sb)], np.int8)
    f["name_exact"] = (g(A, "name_skc", a) == g(B, "name_skc", b)).astype(np.int8)
    f["cand_addr_len"] = np.fromiter((len(s) for s in ya), np.int32, len(ya))
    f["s1_addr_len"] = np.fromiter((len(s) for s in xa), np.int32, len(xa))
    f["cand_name_len"] = np.fromiter((len(s) for s in g(B, "name_b", b)), np.int32, len(b))
    f["s1_name_len"] = np.fromiter((len(s) for s in g(A, "name_b", a)), np.int32, len(a))
    f["cand_nonascii"] = g(B, "nonascii", b)
    f["is_s3"] = np.int8(is_s3)
    # context: how this candidate compares with the other candidates of the same Source 1 record
    v = f.nsk_tset.to_numpy() + f.ask_tset.to_numpy()
    grp = pd.DataFrame({"a": a, "v": v}).groupby("a")["v"]
    f["rank_s1"] = grp.rank(ascending=False, method="min").to_numpy(np.float32)
    f["gap_s1"] = (grp.transform("max").to_numpy() - v).astype(np.float32)
    f["n_cand_s1"] = grp.transform("size").to_numpy(np.int32)
    return f

# ---------------------------------------------------------------------
# 5. Scoring (same formula as the leaderboard) and the final decision rule
# ---------------------------------------------------------------------
def macro_f05(pred, truth, ids):
    sc = []
    for s in ids:
        p, t = pred.get(s, set()), truth.get(s, set())
        if not t:
            sc.append(1.0 if not p else 0.0); continue
        tp = len(p & t)
        if tp == 0:
            sc.append(0.0); continue
        P, R = tp / len(p), tp / len(t)
        sc.append(1.25 * P * R / (0.25 * P + R))
    return float(np.mean(sc))

def decide(pairs, thr):
    """pairs: DataFrame(s1_id, cand_id, p). Keep p >= thr; each S2/S3 record goes to at most one S1 (its best)."""
    d = pairs[pairs.p >= thr].sort_values("p", ascending=False).drop_duplicates("cand_id")
    return d.groupby("s1_id")["cand_id"].apply(set).to_dict()

def write_tsv(path, s1_ids, mapping, col):
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(f"source1_entity_id\t{col}\n")
        for s in s1_ids:
            fh.write(s + "\t" + ",".join(sorted(mapping.get(s, ()))) + "\n")


In [ ]:
# ======================= v6 settings =======================
import shutil
ROOT = os.environ.get("ROOT", "/kaggle/input")      # the notebook searches here for the .tsv files
WORK = os.environ.get("WORK", "/kaggle/working")    # models + outputs are saved here
SMOKE = 0             # 1 = tiny quick test run (only for checking the code)
# Training businesses per country (v1 used 150k in total). More data = better model, but slower.
N_TRAIN = {"India": int(os.environ.get("N_IN", 350_000)), "US": int(os.environ.get("N_US", 500_000))} if not SMOKE else {"India": 4_000, "US": 6_000}
N_GENERAL = int(os.environ.get("N_GEN", 120_000)) if not SMOKE else 2_000          # per country, for the general model used on France
BATCH = int(os.environ.get("BATCH", 100_000))       # businesses processed at once (lower it if memory runs out)
MINUTES_PER_MODEL = {"India": 70, "US": 90, "general": 45}   # safety cap on training time for each model
# France has no answers to tune on, and v1 made many wrong France matches (public board 0.9405 vs 0.9529 on
# validation). So France (and any unseen country) uses a stricter cut-off: the tuned one + this margin.
UNSEEN_MARGIN = 0.15
GENERAL_DROP = ["state_rel", "cand_nonascii"]       # features that mean something different in France
# Big temporary files go to /kaggle/temp when it exists (it does not count toward the 20 GB output limit)
TMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else WORK
os.makedirs(WORK, exist_ok=True); os.makedirs(TMP, exist_ok=True)
log("CPU cores:", N_CPU, "| training businesses:", N_TRAIN, "| smoke test:", bool(SMOKE))


In [ ]:
# ======================= v6 helpers =======================
# Same features as v1 (FEATS). What is new: separate models per country, and a separate
# cut-off per country and per source, tuned for F0.5 on held-out training businesses.
FEATS = ["sim_name", "sim_addr", "nb_tset", "nb_tsort", "nb_ratio", "nb_partial", "nb_jw", "nsk_tset", "nsk_tsort",
         "nsk_ratio", "nsk_partial", "nsk_jw", "nskc_ratio", "nskc_partial", "ask_tset", "ask_ratio", "ask_partial",
         "num_jacc", "num_contain", "state_rel", "name_exact", "cand_addr_len", "s1_addr_len", "cand_name_len",
         "s1_name_len", "cand_nonascii", "is_s3", "rank_s1", "gap_s1", "n_cand_s1",
         # new in v6: how common the name is. Generic names ("Bordeaux Club SARL") are common in India and
         # very common in France, and an exact name match means less when many businesses share the name.
         "s1_name_freq", "cand_name_freq", "n_name_close"]
GRID = np.round(np.arange(0.30, 0.9001, 0.025), 3)     # cut-offs tried
S3_OFFSET = 10 ** 9                                      # candidate key = row in its source (+1e9 for Source 3)

def add_name_freq(D):
    """How many records per 100k in the same country share this cleaned name."""
    g = D.groupby(["country", "name_skc"]).name_skc.transform("size")
    D["name_freq"] = (g / D.groupby("country").name_skc.transform("size") * 1e5).astype(np.float32)
    return D

def pair_features(idx, Ac, Bc, is_s3):
    """Shortlist + v1 features (+ 3 name-commonness features) for a batch of Source 1 rows (Ac)
    against one country of one source (Bc)."""
    P = idx.query(Ac)
    if len(P) == 0:
        return P, None
    F = features(P, Ac, Bc, is_s3)
    a, b = P.a.to_numpy(), P.b.to_numpy()
    F["s1_name_freq"] = Ac.name_freq.to_numpy()[a]
    F["cand_name_freq"] = Bc.name_freq.to_numpy()[b]
    F["n_name_close"] = pd.Series(F.nskc_ratio.to_numpy() >= 90).groupby(a).transform("sum").to_numpy()
    return P, F[FEATS].astype(np.float32)

def f05_rows(npred, tp, ntrue):
    """F0.5 of each Source 1 record from counts (same formula as macro_f05 / the leaderboard)."""
    P = tp / np.maximum(npred, 1); R = tp / np.maximum(ntrue, 1)
    f = np.where(tp > 0, 1.25 * P * R / np.maximum(0.25 * P + R, 1e-12), 0.0)
    return np.where(ntrue == 0, (npred == 0).astype(float), f)

def dedup(V):
    """Each Source 2/3 record goes to at most one Source 1 record: its most likely one."""
    return V.sort_values("p", ascending=False).drop_duplicates("cand").reset_index(drop=True)

def tune_cutoffs(V, rows, ntrue_all):
    """V: deduplicated val pairs (s1, src, p, label). rows: the Source 1 rows to score.
    Tries every (Source 2 cut-off, Source 3 cut-off) pair; returns (best_f05, t2, t3, per-row F0.5)."""
    pos = np.full(len(ntrue_all), -1); pos[rows] = np.arange(len(rows)); n = len(rows)
    V = V[pos[V.s1.to_numpy()] >= 0]
    s, src, p, lab = pos[V.s1.to_numpy()], V.src.to_numpy(), V.p.to_numpy(), V.label.to_numpy().astype(bool)
    cnt = {}
    for k in (0, 1):
        for t in GRID:
            m = (src == k) & (p >= t)
            cnt[k, t] = (np.bincount(s[m], minlength=n), np.bincount(s[m & lab], minlength=n))
    nt = ntrue_all[rows]; best = (-1, None, None, None)
    for t2 in GRID:
        for t3 in GRID:
            f = f05_rows(cnt[0, t2][0] + cnt[1, t3][0], cnt[0, t2][1] + cnt[1, t3][1], nt)
            if f.mean() > best[0]:
                best = (f.mean(), float(t2), float(t3), f)
    return best

def single_cutoff(V, rows, ntrue_all, t):
    pos = np.full(len(ntrue_all), -1); pos[rows] = np.arange(len(rows))
    V = V[(pos[V.s1.to_numpy()] >= 0) & (V.p.to_numpy() >= t)]
    s = pos[V.s1.to_numpy()]
    return f05_rows(np.bincount(s, minlength=len(rows)),
                    np.bincount(s[V.label.to_numpy().astype(bool)], minlength=len(rows)), ntrue_all[rows])

def time_limit(minutes, name):
    """Stops LightGBM when its time budget is used up, and logs progress every 100 trees."""
    t0 = time.time()
    def cb(env):
        if env.iteration % 100 == 0 and env.evaluation_result_list:
            log(f"  {name}: tree {env.iteration} val logloss {env.evaluation_result_list[0][2]:.5f}")
        if time.time() - t0 > minutes * 60:
            log(f"  {name}: time budget reached at tree {env.iteration}")
            raise lgb.callback.EarlyStopException(env.iteration, env.evaluation_result_list)
    cb.order = 40
    return cb

LGB_PARAMS = dict(objective="binary", learning_rate=0.1, num_leaves=255, min_data_in_leaf=200,   # faster learning: 6x more data than v1
                  bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.8, num_threads=N_CPU,
                  verbose=-1, seed=6)


In [ ]:
# ======================= v6 training =======================
# 1) pick training businesses per country + the SAME 30k held-out businesses v1 was checked on
# 2) build shortlist + features batch by batch, saving each batch to disk (so memory stays low)
# 3) train one LightGBM per country (India, US) and one general model (for France / unseen countries)
# 4) tune a cut-off per country and per source (Source 2 vs Source 3) for F0.5 on the held-out businesses
T0 = time.time()
log("loading train Source 1 + ground truth")
s1 = load(ROOT, "train_source1"); gt = load(ROOT, "train_ground_truth")
v1s = s1.sample(min(150_000, len(s1)), random_state=0).reset_index(drop=True)     # exactly v1's sample ...
val_ids = set(np.random.default_rng(42).choice(v1s.entity_id.to_numpy(), int(0.2 * len(v1s)), replace=False))  # ... and v1's check set
rest = s1[~s1.entity_id.isin(val_ids)]
parts = [s1[s1.entity_id.isin(val_ids)]]
for c, n in N_TRAIN.items():
    pool = rest[rest.country == c]
    parts.append(pool.sample(min(n, len(pool)), random_state=6))
A = pd.concat(parts, ignore_index=True); del s1, rest, parts, v1s; gc.collect()
is_val = A.entity_id.isin(val_ids).to_numpy()
# general model: the first N_GENERAL training businesses of each country (already in random order)
gen = np.zeros(len(A), bool)
for c in N_TRAIN:
    gen[np.flatnonzero((A.country.to_numpy() == c) & ~is_val)[:N_GENERAL]] = True
truth = {r.source1_entity_id: set(x for x in r.matched_entity_ids.split(",") if x) for r in gt.itertuples()}
del gt; gc.collect()
truth_rows = [truth.get(s, set()) for s in A.entity_id]
ntrue = np.array([len(t) for t in truth_rows]); del truth
A = add_name_freq(prepare(A)); log("prepared Source 1:", len(A), "businesses,", int(is_val.sum()), "held out for checking")
print(A[~is_val].country.value_counts().to_string())

PAIRS = f"{TMP}/v6_train_pairs"; os.makedirs(PAIRS, exist_ok=True)
n_true_found = 0
for src, is_s3 in (("train_source2", 0), ("train_source3", 1)):
    log("loading", src); B = add_name_freq(prepare(load(ROOT, src))); log("prepared", src, B.shape)
    bids = B.entity_id.to_numpy()
    for country in A.country.unique():
        a_rows = np.flatnonzero(A.country.to_numpy() == country)
        b_rows = np.flatnonzero(B.country.to_numpy() == country)
        if len(b_rows) == 0: continue
        Bc = B.iloc[b_rows].reset_index(drop=True); idx = Index(Bc)
        nb = -(-len(a_rows) // BATCH)
        for i in range(0, len(a_rows), BATCH):
            out = f"{PAIRS}/{src}__{country}__{i}.parquet"
            if os.path.exists(out): continue
            rows = a_rows[i:i + BATCH]; Ac = A.iloc[rows].reset_index(drop=True)
            P, F = pair_features(idx, Ac, Bc, is_s3)
            if F is None: continue
            s1r = rows[P.a.to_numpy()]; bg = b_rows[P.b.to_numpy()]
            F["s1"] = s1r.astype(np.int32)
            F["cand"] = bg.astype(np.int64) + is_s3 * S3_OFFSET
            F["label"] = np.fromiter((c in truth_rows[r] for r, c in zip(s1r, bids[bg])), bool, len(bg)).astype(np.int8)
            F.to_parquet(out, compression="zstd")
            log(src, country, f"batch {i // BATCH + 1}/{nb}", "pairs", len(F), "true matches found", int(F.label.sum()))
            del P, F; gc.collect()
        del idx, Bc; gc.collect()
    del B, bids; gc.collect()
n_found = pl.scan_parquet(f"{PAIRS}/*.parquet").select(pl.col("label").sum()).collect().item()
n_pairs = pl.scan_parquet(f"{PAIRS}/*.parquet").select(pl.len()).collect().item()
log("all pairs:", n_pairs, "| per business", round(n_pairs / len(A), 1),
    "| shortlist recall", round(n_found / ntrue.sum(), 4), "| minutes so far", round((time.time() - T0) / 60))

def load_pairs(keep_rows):
    """Reads the saved pairs of the given Source 1 rows."""
    keep = pl.DataFrame({"s1": np.flatnonzero(keep_rows).astype(np.int32)})
    return pl.scan_parquet(f"{PAIRS}/*.parquet").join(keep.lazy(), on="s1", how="semi").collect()

def train_one(name, keep_rows, feats):
    D = load_pairs(keep_rows)
    s = D["s1"].to_numpy(); va = is_val[s]
    X = D.select(feats).to_numpy().astype(np.float32, copy=False); y = D["label"].to_numpy()
    meta_v = pd.DataFrame({"s1": s[va], "cand": D["cand"].to_numpy()[va], "label": y[va]})
    meta_v["src"] = (meta_v.cand.to_numpy() >= S3_OFFSET).astype(np.int8)
    del D; gc.collect()
    Xv, yv = X[va], y[va]
    log(f"{name} model: {int((~va).sum())} training pairs, {int(va.sum())} check pairs")
    dtr = lgb.Dataset(X[~va], y[~va], feature_name=feats, free_raw_data=True).construct()
    del X, y; gc.collect()
    dva = lgb.Dataset(Xv, yv, reference=dtr)
    bst = lgb.train(LGB_PARAMS, dtr, num_boost_round=5000, valid_sets=[dva],
                    callbacks=[lgb.early_stopping(100, verbose=False), time_limit(MINUTES_PER_MODEL[name], name)])
    log(f"{name} model: {bst.best_iteration} trees")
    bst.save_model(f"{WORK}/v6_model_{name}.txt", num_iteration=bst.best_iteration)
    meta_v["p"] = bst.predict(Xv, num_iteration=bst.best_iteration, num_threads=N_CPU)
    del dtr, dva, Xv, yv; gc.collect()
    imp = pd.Series(bst.feature_importance(), feats).sort_values(ascending=False)
    log(f"{name} top features: " + ", ".join(imp.index[:8]))
    return dedup(meta_v)

countries = list(N_TRAIN)
val_rows = np.flatnonzero(is_val)
meta = {"features": {}, "cutoffs": {}, "val_f05": {}}
per_row_f = np.zeros(len(A))
val_country = {}
for c in countries:
    rows_c = (A.country.to_numpy() == c)
    V = train_one(c, rows_c, FEATS)
    vr = np.flatnonzero(rows_c & is_val); val_country[c] = (V, vr)
    f, t2, t3, frow = tune_cutoffs(V, vr, ntrue)
    per_row_f[vr] = frow
    meta["features"][c] = FEATS; meta["cutoffs"][c] = {"0": t2, "1": t3}; meta["val_f05"][c] = f
    log(f"BEST val F0.5 {c}: {f:.4f}  (cut-off Source 2 = {t2}, Source 3 = {t3}; {len(vr)} businesses)")

GFEATS = [f for f in FEATS if f not in GENERAL_DROP]
Vg = train_one("general", gen | is_val, GFEATS)
fg, g2, g3, _ = tune_cutoffs(Vg, val_rows, ntrue)
meta["features"]["general"] = GFEATS; meta["cutoffs"]["general"] = {"0": g2, "1": g3}; meta["val_f05"]["general"] = fg
log(f"general model (used for France): val F0.5 on US+India {fg:.4f}  (cut-off Source 2 = {g2}, Source 3 = {g3})")
_Vg = Vg[Vg.p.to_numpy() >= np.where(Vg.src.to_numpy() == 1, min(g3 + UNSEEN_MARGIN, 0.95), min(g2 + UNSEEN_MARGIN, 0.95))]
_s = _Vg.s1.to_numpy(); _l = _Vg.label.to_numpy().astype(bool)
_f = f05_rows(np.bincount(_s, minlength=len(A)), np.bincount(_s[_l], minlength=len(A)), ntrue)[val_rows].mean()
log(f"general model with the stricter France cut-off (+{UNSEEN_MARGIN}): val F0.5 on US+India {_f:.4f} (the price of playing safe)")

overall = per_row_f[val_rows].mean(); meta["val_f05"]["overall"] = overall
# for reference: the country models with one shared cut-off, like v1
Vall = pd.concat([v for v, _ in val_country.values()], ignore_index=True)
single = {t: single_cutoff(Vall, val_rows, ntrue, t).mean() for t in GRID}
t_single = max(single, key=single.get)
log(f"country models with one shared cut-off {t_single}: {single[t_single]:.4f}")
V_all_pairs = pd.concat([v for v, _ in val_country.values()], ignore_index=True)
ceiling = f05_rows(np.bincount(V_all_pairs.s1[V_all_pairs.label == 1], minlength=len(A)),
                   np.bincount(V_all_pairs.s1[V_all_pairs.label == 1], minlength=len(A)), ntrue)[val_rows].mean()
log(f"ceiling (perfect model on this shortlist): {ceiling:.4f}")
json.dump(meta, open(f"{WORK}/v6_meta.json", "w"), indent=1)
log(f"BEST val F0.5 overall: {overall:.4f}   (v1 on the same {len(val_rows)} businesses: 0.9529)")
for c in countries: log(f"BEST val F0.5 {c}: {meta['val_f05'][c]:.4f}")
log("training finished in", round((time.time() - T0) / 60), "minutes")
del A, truth_rows, val_country, Vall, V_all_pairs, Vg; gc.collect()
shutil.rmtree(PAIRS, ignore_errors=True)


In [ ]:
# ======================= v6 test prediction =======================
# India -> India model, US -> US model, France (or any other country) -> general model.
# Each batch is saved to disk, so memory stays low and a crashed run can resume.
meta = json.load(open(f"{WORK}/v6_meta.json"))
boosters = {k: lgb.Booster(model_file=f"{WORK}/v6_model_{k}.txt") for k in meta["features"]}
PARTS = f"{TMP}/parts_v6"; os.makedirs(PARTS, exist_ok=True)
log("loading test Source 1"); A = add_name_freq(prepare(load(ROOT, "test_source1"))); log("prepared", A.shape)
print(A.country.value_counts().to_string())
s1_ids = A.entity_id.to_numpy()
for src, is_s3 in (("test_source2", 0), ("test_source3", 1)):
    todo = [(c, i) for c in A.country.unique() for i in range(0, int((A.country == c).sum()), BATCH)
            if not os.path.exists(f"{PARTS}/{src}__{c}__{i}.parquet")]
    if not todo:
        log(src, "already done"); continue
    log("loading", src); B = add_name_freq(prepare(load(ROOT, src))); log("prepared", src, B.shape)
    for country in A.country.unique():
        key = country if country in boosters else "general"
        thr = meta["cutoffs"][key][str(is_s3)]
        if key == "general":
            thr = min(thr + UNSEEN_MARGIN, 0.95)          # stricter for France / unseen countries
        a_rows = np.flatnonzero(A.country.to_numpy() == country)
        Bc = B[B.country == country].reset_index(drop=True)
        if len(Bc) == 0 or not any(c == country for c, _ in todo): continue
        idx = Index(Bc); bids = Bc.entity_id.to_numpy()
        for i in range(0, len(a_rows), BATCH):
            out = f"{PARTS}/{src}__{country}__{i}.parquet"
            if os.path.exists(out): continue
            rows = a_rows[i:i + BATCH]; Ac = A.iloc[rows].reset_index(drop=True)
            P, F = pair_features(idx, Ac, Bc, is_s3)
            if F is None:
                P = pd.DataFrame({"a": np.zeros(0, int), "b": np.zeros(0, int)}); p = np.zeros(0, np.float32)
            else:
                p = boosters[key].predict(F[meta["features"][key]].to_numpy(np.float32), num_threads=N_CPU).astype(np.float32)
            pd.DataFrame({"s1_row": rows[P.a.to_numpy()].astype(np.int32), "cand_id": bids[P.b.to_numpy()],
                          "p": p, "thr": np.float32(thr)}).to_parquet(out)
            log(src, country, f"({key} model)", f"batch {i // BATCH + 1}/{-(-len(a_rows) // BATCH)}",
                "pairs", len(p), "kept", int((p >= thr).sum()))
            del P, F, p; gc.collect()
        del idx, Bc; gc.collect()
    del B; gc.collect()

# ---- combine, using little memory (loading all ~150M candidate pairs at once needs over 14 GB) ----
log("combining batches")
# 1) final matches: only pairs above their cut-off are loaded (a small share of all pairs)
best = (pl.scan_parquet(f"{PARTS}/*.parquet").filter(pl.col("p") >= pl.col("thr")).collect()
        .sort("p", descending=True).unique(subset="cand_id", keep="first", maintain_order=True))  # each S2/S3 record -> its best S1
match = best.group_by("s1_row").agg(pl.col("cand_id").sort())
mline = np.full(len(s1_ids), "", dtype=object); nmatch = np.zeros(len(s1_ids), np.int32)
for r, c in zip(match["s1_row"].to_list(), match["cand_id"].to_list()):
    mline[r] = ",".join(c); nmatch[r] = len(c)
del best, match; gc.collect()
os.makedirs(f"{WORK}/output", exist_ok=True)
with open(f"{WORK}/output/matching_results.tsv", "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")
    for s, m in zip(s1_ids, mline):
        f.write(s + "\t" + m + "\n")
del mline; gc.collect()
# 2) candidate list, one batch at a time (the Source 2 and Source 3 files of a batch cover the same S1 rows)
cline = np.full(len(s1_ids), "", dtype=object)
files = sorted(glob.glob(f"{PARTS}/*.parquet"))
twins = {}
for f in files:
    twins.setdefault(os.path.basename(f).split("__", 1)[1], []).append(f)
for group in twins.values():
    g = pl.concat([pl.read_parquet(x, columns=["s1_row", "cand_id"]) for x in group]) \
          .group_by("s1_row").agg(pl.col("cand_id").unique(maintain_order=True))
    for r, c in zip(g["s1_row"].to_list(), g["cand_id"].to_list()):
        cline[r] = ",".join(c)
    del g; gc.collect()
with open(f"{WORK}/output/candidate_pairs.tsv", "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tcandidate_entity_ids\n")
    for s, c in zip(s1_ids, cline):
        f.write(s + "\t" + c + "\n")
del cline; gc.collect()
for c in A.country.unique():
    rr = A.country.to_numpy() == c
    log(f"  {c}: {int(rr.sum())} businesses, avg matches {nmatch[rr].mean():.2f}, empty {(nmatch[rr] == 0).mean():.1%}")
n = int((nmatch > 0).sum())
log("done:", len(s1_ids), "rows,", n, "with matches,", len(s1_ids) - n, "empty; avg matches", round(nmatch.mean(), 2))


In [ ]:
validator = glob.glob(f'{ROOT}/**/validate_submission.py', recursive=True)[0]
test_dir = os.path.dirname(find_file(ROOT, 'test_source1')[0])
# checks the matching file only: checking the 150M-pair candidate file too runs out of memory
!python3 {validator} --matching {WORK}/output/matching_results.tsv --candidate {WORK}/no_candidate_check.tsv --test-dir {test_dir}

**Done.** Open this version's **Output** tab, go into `output/`, and download **`matching_results.tsv`**. Keep `candidate_pairs.tsv` for the final zip.